# Réemploi des plaques gravées entre éditeurs

Certains jeux de plaques gravées (identifiés par le graveur, ex. "Anonyme1572") sont réutilisés
d'une édition à l'autre : soit repris par le même éditeur des années plus tard (réédition avec
les mêmes gravures), soit transmis à un autre éditeur (achat des plaques, héritage...). Exemple :
"Anonyme1572" est publié par Francesco de' Franceschi (Venise, 1572), puis les mêmes plaques
réapparaissent chez Pietro Deuchino en 1587 et 1588.

Cette frise trace, pour chaque jeu de plaques réemployé, la chronologie de ses éditeurs — sur
l'ensemble du corpus (19 villes), pas seulement Lyon/Paris/Venise comme dans
`02_nuage_editions_villes.ipynb`.

Même source de données que `01_carte_circulation.ipynb` et `02_nuage_editions_villes.ipynb` :
`retours_celine/BNU_corpus.ods` (feuille `Synthèse`).

In [ ]:
import os
import re
import json
from odf.opendocument import load as charger_ods
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

RACINE = os.path.abspath("../..")
DOSSIER_VIZ = os.path.join(RACINE, "resultats", "Datavis")
os.makedirs(DOSSIER_VIZ, exist_ok=True)
CHEMIN_SORTIE = os.path.join(DOSSIER_VIZ, "reemploi_plaques.html")
CHEMIN_CORPUS = os.path.join(RACINE, "retours_celine", "BNU_corpus.ods")

## 1. Chargement du corpus

Même lecteur ODS cellule par cellule que les deux autres notebooks (voir
`01_carte_circulation.ipynb` pour le détail des limites de `pandas.read_excel` sur ce fichier).
Contrairement à `02_nuage_editions_villes.ipynb`, on garde ici **toutes** les villes du corpus :
ce n'est pas une question géographique, c'est une question de filiation entre éditeurs à
travers le temps. Seules les éditions avec un graveur identifiable sont retenues — sans
identité de plaques, il n'y a rien à tracer.

In [ ]:
def lire_feuille_ods(chemin, nom_feuille):
    """Lit une feuille ODS cellule par cellule (pandas ignore des colonnes de ce fichier)."""
    doc = charger_ods(chemin)
    table = next(t for t in doc.spreadsheet.getElementsByType(Table)
                 if t.getAttribute("name") == nom_feuille)
    lignes_brutes = table.getElementsByType(TableRow)

    def valeurs_ligne(ligne):
        valeurs, col = {}, 0
        for cellule in ligne.getElementsByType(TableCell):
            rep = cellule.getAttribute("numbercolumnsrepeated")
            rep = int(rep) if rep else 1
            paras = cellule.getElementsByType(P)
            texte = " ".join(teletype.extractText(p) for p in paras)
            for k in range(rep):
                valeurs[col + k] = texte
            col += rep
        return valeurs

    entetes = valeurs_ligne(lignes_brutes[0])
    colonnes = {i: t.strip() for i, t in entetes.items() if t.strip()}

    lignes = []
    for ligne in lignes_brutes[1:]:
        rep = ligne.getAttribute("numberrowsrepeated")
        rep = int(rep) if rep else 1
        valeurs = valeurs_ligne(ligne)
        if not any(v.strip() for v in valeurs.values()):
            continue  # ligne vide (fin de feuille)
        d = {nom: valeurs.get(i, "").strip() for i, nom in colonnes.items()}
        lignes.extend([d] * rep)
    return lignes

def extraire_annee(valeur):
    """Renvoie la première année à 4 chiffres trouvée (ex: "1527 / 1528 ?" -> 1527)."""
    m = re.search(r"\d{4}", str(valeur))
    return int(m.group()) if m else None

def extraire_lien(row):
    """Choisit le premier lien exploitable, par ordre de préférence (même logique que
    01_carte_circulation.ipynb et 02_nuage_editions_villes.ipynb)."""
    for col in ["version numérisée 1", "version numérisée 2", "Biblioteca Digital Ovidiana", "url catalogue"]:
        val = row.get(col, "")
        if not val:
            continue
        premier = val.split(";")[0].strip()
        if premier.startswith("http"):
            return premier
    return None

def graveur_ou_inconnu(g):
    """None pour un graveur non identifié ('?', 'inaccessible', case vide) : ces éditions sont
    écartées plus bas, contrairement à 02_nuage_editions_villes.ipynb qui leur donnait un repli
    textuel — ici pas d'identité de plaques, pas de ligne dans la frise."""
    g = (g or "").strip()
    if not g or g.lower() in {"?", "inaccessible"}:
        return None
    return g

def categorie_technique(t):
    t = (t or "").strip().lower()
    if t == "bois":
        return "bois"
    if t == "cuivre":
        return "cuivre"
    return "inconnue"  # vide, "inaccessible", "?", etc. -> repli neutre plutôt que fragmenter

COL_GRAVEUR = "graveur\xa0: Nom, Prénom"

corpus = lire_feuille_ods(CHEMIN_CORPUS, "Synthèse")

editions = []
for row in corpus:
    annee = extraire_annee(row.get("année", ""))
    if annee is None:
        continue
    graveur = graveur_ou_inconnu(row.get(COL_GRAVEUR, ""))
    if graveur is None:
        continue
    titre = row.get("titre abrégé") or row.get("titre complet") or ""
    editions.append({
        "ville": row.get("ville", "").strip() or "Ville inconnue",
        "annee": annee,
        "titre": titre.strip(),
        "technique": categorie_technique(row.get("technique", "")),
        "graveur": graveur,
        "publisher": row.get("publisher", "").strip() or "Éditeur non identifié",
        "lien": extraire_lien(row),
    })

print(len(editions), "éditions avec un graveur identifiable, sur", len(corpus), "au total")
print(len({e['graveur'] for e in editions}), "jeux de plaques distincts")

## 2. Choix de visualisation

- **Forme** : une frise en couloirs ("swimlanes") plutôt qu'un graphe en réseau à disposition
  libre. Une ligne horizontale par jeu de plaques réemployé (axe des années partagé,
  1490-1750), un point par édition l'utilisant, relié chronologiquement au suivant. Un vrai
  graphe en réseau rendrait mieux les cas très enchevêtrés (une plaque qui circule entre
  beaucoup d'éditeurs dans tous les sens), mais demanderait une disposition physique
  (force-directed) à construire de zéro ; ici les chaînes restent courtes (2 à 11 éditions),
  la frise reste lisible et réutilise le code de frise chronologique déjà écrit pour les deux
  autres notebooks.
- **Seuil** : seuls les jeux de plaques utilisés dans **au moins 2 éditions** sont affichés —
  une plaque utilisée une seule fois n'a pas de "vie" à raconter.
- **Ordre des lignes** : par nombre d'éditions décroissant (les jeux de plaques les plus
  réemployés en premier, les cas les plus intéressants remontent naturellement), puis par
  année de première apparition.
- **Type de segment entre deux éditions consécutives** — c'est le cœur de cette visualisation :
  - **réimpression** (même éditeur des deux côtés) : trait fin neutre.
  - **transmission** (éditeur différent) : trait coloré avec une flèche, dans le sens
    chronologique (comme les flèches de circulation de `01_carte_circulation.ipynb`, mais ici
    ce n'est pas la plaque qui se déplace géographiquement, c'est sa propriété qui change de
    main).
  - **incertain** : trait pointillé, dès que l'éditeur d'un côté ou de l'autre n'est pas
    identifié ("s.n.", case vide) — impossible de dire s'il s'agit du même éditeur ou d'un
    autre, donc pas de conclusion hâtive (même principe que les mentions ambiguës écartées
    dans les flèches de copie de `01_carte_circulation.ipynb`).
- **Interaction** : identique aux deux autres notebooks — survol = aperçu rapide sans lien,
  clic = infobulle épinglée avec le détail et le lien "voir" (le survol seul ne permettait pas
  d'atteindre le lien avant qu'il disparaisse). Tableau détaillé dépliable pour l'accessibilité.

In [ ]:
groupes_plaques = {}
for e in editions:
    groupes_plaques.setdefault(e["graveur"], []).append(e)

plaques_reemployees = {
    g: sorted(u, key=lambda e: e["annee"])
    for g, u in groupes_plaques.items() if len(u) > 1
}

def editeur_fiable(pub):
    """Un éditeur non identifié ('s.n.', case vide) ne permet pas de dire avec certitude si
    deux éditions se succèdent chez le même éditeur ou changent de main."""
    return pub not in {"s.n.", "Éditeur non identifié"}

def type_segment(pub1, pub2):
    if not (editeur_fiable(pub1) and editeur_fiable(pub2)):
        return "incertain"
    return "reprise" if pub1 == pub2 else "transfert"

ordre_plaques = sorted(
    plaques_reemployees.items(),
    key=lambda kv: (-len(kv[1]), kv[1][0]["annee"])
)

plaques = []
for rang, (graveur, eds) in enumerate(ordre_plaques):
    segments = [
        {"an1": a["annee"], "an2": b["annee"], "type": type_segment(a["publisher"], b["publisher"])}
        for a, b in zip(eds, eds[1:])
    ]
    plaques.append({"graveur": graveur, "rang": rang, "editions": eds, "segments": segments})

nb_transferts = sum(1 for p in plaques for s in p["segments"] if s["type"] == "transfert")
nb_reprises = sum(1 for p in plaques for s in p["segments"] if s["type"] == "reprise")
nb_incertains = sum(1 for p in plaques for s in p["segments"] if s["type"] == "incertain")

print(len(groupes_plaques), "jeux de plaques au total —", len(plaques), "réemployés (≥2 éditions), affichés ci-dessous")
print(nb_transferts, "transmissions à un autre éditeur,", nb_reprises, "réimpressions par le même éditeur,",
      nb_incertains, "cas incertains (éditeur non identifié d'un côté ou de l'autre)")
print()
for p in plaques[:10]:
    chaine = " → ".join(f"{e['publisher']} ({e['annee']})" for e in p["editions"])
    print(f"  {p['graveur']:30s} {chaine}")

## 3. Génération de la frise (HTML autonome)

Une seule grande frise SVG (pas de carte cette fois : la question n'est pas géographique),
une ligne par jeu de plaques réemployé, un bandeau alterné ("zèbre") pour repérer facilement
une ligne d'une autre. Légende des trois types de segment, infobulle au survol/clic identique
aux deux autres notebooks, et un tableau détaillé dépliable listant toutes les éditions de
chaque jeu de plaques dans l'ordre chronologique.

In [ ]:
LIBELLES_TECHNIQUE = {"bois": "Bois", "cuivre": "Cuivre", "inconnue": "Technique inconnue"}

FRISE_ANNEE_MIN, FRISE_ANNEE_MAX = 1490, 1750
FRISE_TICKS = [1500, 1600, 1700]
LARGEUR = 1000
MARGE = {"gauche": 210, "droite": 20, "haut": 30, "bas": 36}
HAUTEUR_LIGNE = 24
HAUTEUR_PLOT = len(plaques) * HAUTEUR_LIGNE
HAUTEUR = MARGE["haut"] + HAUTEUR_PLOT + MARGE["bas"]

def frise_x(annee):
    t = (annee - FRISE_ANNEE_MIN) / (FRISE_ANNEE_MAX - FRISE_ANNEE_MIN)
    return MARGE["gauche"] + t * (LARGEUR - MARGE["gauche"] - MARGE["droite"])

def y_ligne(rang):
    return MARGE["haut"] + rang * HAUTEUR_LIGNE + HAUTEUR_LIGNE / 2

# --- Construction du SVG : bandes zébrées + étiquettes, grille des années, segments, points ---
# (dans cet ordre, pour que les segments passent par-dessus les bandes/la grille, et les
# points par-dessus les segments)
elements_svg = []

for p in plaques:
    y0 = MARGE["haut"] + p["rang"] * HAUTEUR_LIGNE
    if p["rang"] % 2 == 1:
        elements_svg.append(f'<rect x="0" y="{y0}" width="{LARGEUR}" height="{HAUTEUR_LIGNE}" class="bande-zebra"/>')
    elements_svg.append(
        f'<text x="8" y="{y_ligne(p["rang"]) + 4:.1f}" class="etiquette-plaque">{p["graveur"]}</text>'
    )

for t in FRISE_TICKS:
    x = frise_x(t)
    elements_svg.append(f'<line x1="{x:.1f}" y1="{MARGE["haut"]}" x2="{x:.1f}" y2="{MARGE["haut"] + HAUTEUR_PLOT:.1f}" class="grille-frise"/>')
    elements_svg.append(f'<text x="{x:.1f}" y="{MARGE["haut"] + HAUTEUR_PLOT + 18:.1f}" text-anchor="middle" class="etiquette-annee-frise">{t}</text>')

for p in plaques:
    y = y_ligne(p["rang"])
    for s in p["segments"]:
        x1, x2 = frise_x(s["an1"]), frise_x(s["an2"])
        marqueur = ' marker-end="url(#fleche-transfert)"' if s["type"] == "transfert" else ""
        elements_svg.append(
            f'<line x1="{x1:.1f}" y1="{y:.1f}" x2="{x2:.1f}" y2="{y:.1f}" class="segment-{s["type"]}"{marqueur}/>'
        )

# `points` : une entrée par édition affichée, dans l'ordre où les cercles sont ajoutés au SVG
# (l'attribut data-i de chaque cercle est son index dans cette liste) — même mécanisme
# d'infobulle par survol/clic que dans 02_nuage_editions_villes.ipynb.
points = []
for p in plaques:
    y = y_ligne(p["rang"])
    for e in p["editions"]:
        i = len(points)
        points.append({
            "fx": round(frise_x(e["annee"]), 1),
            "fy": round(y, 1),
            "graveur": p["graveur"],
            "annee": e["annee"],
            "titre": e["titre"],
            "ville": e["ville"],
            "editeur": e["publisher"],
            "technique": e["technique"],
            "lien": e["lien"],
        })
        elements_svg.append(
            f'<circle class="point-frise" data-i="{i}" cx="{points[-1]["fx"]}" cy="{points[-1]["fy"]}" r="4.5"/>'
        )

FRISE_SVG_CONTENU = "\n".join(elements_svg)
print(len(points), "points affichés,", HAUTEUR, "px de haut")

# --- Tableau détaillé : une ligne par édition, groupée par jeu de plaques (même ordre que la frise) ---
def lignes_tableau_plaque(p):
    lignes = []
    for j, e in enumerate(p["editions"]):
        lien_html = f'<a href="{e["lien"]}" target="_blank">voir</a>' if e["lien"] else ""
        graveur_cell = p["graveur"] if j == 0 else ""
        lignes.append(
            f'<tr><td>{graveur_cell}</td><td>{e["annee"]}</td><td>{e["ville"]}</td>'
            f'<td>{e["publisher"]}</td><td>{e["titre"]}</td>'
            f'<td>{LIBELLES_TECHNIQUE[e["technique"]]}</td><td>{lien_html}</td></tr>'
        )
    return lignes

LIGNES_TABLEAU = "\n".join(l for p in plaques for l in lignes_tableau_plaque(p))

In [ ]:
TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="utf-8">
<title>Réemploi des plaques gravées entre éditeurs</title>
<style>
  :root {
    --surface: #fffaf0; --texte-fort: #2b1e15; --texte-att: #6b5c4f; --trait: #d8cfc0;
    --contour-point: #3e2c23; --c-lien: #2a78d6; --c-transfert: #c0392b; --bande: #f2e9d8;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --surface: #1a1a19; --texte-fort: #f2ece2; --texte-att: #c3baa9; --trait: #3a352c;
      --contour-point: #f2ece2; --c-lien: #3987e5; --c-transfert: #e0685a; --bande: #232019;
    }
  }
  body { margin:0; font-family:Georgia,serif; background:var(--surface); color:var(--texte-fort); }
  .page { max-width:1080px; margin:0 auto; padding:16px 20px 32px; position:relative; }
  h1 { font-size:19px; margin:0 0 4px; }
  p.souschapo { font-size:13px; color:var(--texte-att); margin:0 0 12px; }

  .legende { display:flex; gap:18px; flex-wrap:wrap; font-size:12px; margin:0 0 14px; }
  .legende .item { display:flex; align-items:center; gap:6px; }
  .legende .trait { display:inline-block; width:26px; height:0; border-top-width:2px; }
  .legende .reprise { border-top:2px solid var(--texte-att); }
  .legende .transfert { border-top:2px solid var(--c-transfert); }
  .legende .incertain { border-top:2px dashed var(--texte-att); }

  .cadre-frise { overflow-x:auto; border:1px solid var(--trait); border-radius:6px;
    box-shadow:0 1px 6px rgba(0,0,0,.15); }
  .frise { display:block; min-width:100%; }
  .bande-zebra { fill:var(--bande); }
  .etiquette-plaque { font-size:11px; fill:var(--texte-fort); }
  .grille-frise { stroke:var(--trait); stroke-width:1; }
  .etiquette-annee-frise { font-size:10px; fill:var(--texte-att); }
  .segment-reprise { stroke:var(--texte-att); stroke-width:1.5; }
  .segment-transfert { stroke:var(--c-transfert); stroke-width:2; }
  .segment-incertain { stroke:var(--texte-att); stroke-width:1.5; stroke-dasharray:3,3; }
  .point-frise { fill:var(--surface); stroke:var(--contour-point); stroke-width:1.3; cursor:pointer;
    transition:r .15s; }
  .point-frise:hover, .point-frise.actif { r:7; fill:var(--c-lien); }

  .action-tableau { margin:14px 0 0; }
  button.bascule { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  table.tableau-detaille { width:100%; border-collapse:collapse; font-size:12px; margin:8px 0;
    display:none; }
  table.tableau-detaille.visible { display:table; }
  table.tableau-detaille th, table.tableau-detaille td { text-align:left; padding:4px 8px;
    border-bottom:1px solid var(--trait); }
  table.tableau-detaille a { color:var(--c-lien); }

  .infobulle { position:absolute; pointer-events:none; background:var(--surface);
    border:1px solid var(--contour-point); border-radius:5px; padding:6px 10px; font-size:12px;
    max-width:260px; opacity:0; transition:opacity .1s; box-shadow:0 2px 8px rgba(0,0,0,.3); z-index:2000; }
  .infobulle.epinglee { pointer-events:auto; }
  .infobulle a { color:var(--c-lien); }
  .infobulle .fermer-infobulle { position:absolute; top:2px; right:6px; cursor:pointer;
    color:var(--texte-att); font-size:13px; }
</style></head><body>
<div class="page">
  <h1>Réemploi des plaques gravées entre éditeurs</h1>
  <p class="souschapo">Une ligne = un jeu de plaques réemployé (≥2 éditions), un point = une
    édition qui l'utilise, positionnée par année. Cliquer un point affiche le détail avec le
    lien "voir" (le survol seul n'affiche qu'un aperçu).</p>
  <div class="legende">
    <div class="item"><span class="trait reprise"></span>réimpression (même éditeur)</div>
    <div class="item"><span class="trait transfert"></span>transmission à un autre éditeur</div>
    <div class="item"><span class="trait incertain"></span>incertain (éditeur non identifié)</div>
  </div>
  <div class="cadre-frise">
    <svg class="frise" viewBox="0 0 __LARGEUR__ __HAUTEUR__" width="__LARGEUR__" height="__HAUTEUR__">
      <defs>
        <marker id="fleche-transfert" viewBox="0 0 10 10" refX="8" refY="5"
          markerWidth="6" markerHeight="6" orient="auto-start-reverse">
          <path d="M0,0 L10,5 L0,10 z" fill="var(--c-transfert)"/>
        </marker>
      </defs>
      __FRISE_SVG__
    </svg>
  </div>
  <div class="action-tableau">
    <button class="bascule" id="boutonTableau">Afficher le tableau détaillé</button>
    <table class="tableau-detaille" id="tableauDetaille">
      <thead><tr><th>Plaques</th><th>Année</th><th>Ville</th><th>Éditeur</th><th>Titre</th><th>Technique</th><th>Lien</th></tr></thead>
      <tbody>
        __LIGNES_TABLEAU__
      </tbody>
    </table>
  </div>
  <div class="infobulle" id="infobulle"></div>
</div>
<script>
  const points = __POINTS__;
  const libellesTechnique = __LIBELLES__;

  // Deux variantes du contenu, comme dans 02_nuage_editions_villes.ipynb : l'aperçu au
  // survol n'inclut PAS le lien (il ne serait pas cliquable, l'infobulle de survol ignorant
  // les clics tant qu'elle n'est pas épinglée) ; la version détaillée, affichée au clic une
  // fois épinglée, inclut le lien "voir".
  function contenuApercu(p) {
    return '<b>' + p.graveur + '</b><br>' +
      '<span>' + p.ville + ', ' + p.annee + ' · ' + libellesTechnique[p.technique] + '</span>' +
      '<br><i>' + p.editeur + '</i>' +
      (p.titre ? '<br>' + p.titre : '');
  }
  function contenuDetaille(p) {
    const lien = p.lien ? '<br><a href="' + p.lien + '" target="_blank">→ voir</a>' : '';
    return contenuApercu(p) + lien;
  }

  const infobulle = document.getElementById('infobulle');
  const page = document.querySelector('.page');
  let infobulleEpinglee = false;

  function positionnerInfobulle(ev) {
    const r = page.getBoundingClientRect();
    infobulle.style.left = (ev.clientX - r.left + 14) + 'px';
    infobulle.style.top = (ev.clientY - r.top + 14) + 'px';
  }
  function fermerInfobulle() {
    infobulleEpinglee = false;
    infobulle.classList.remove('epinglee');
    infobulle.style.opacity = 0;
    document.querySelectorAll('.point-frise.actif').forEach(c => c.classList.remove('actif'));
  }

  document.querySelectorAll('.point-frise').forEach(cercle => {
    const p = points[+cercle.dataset.i];
    cercle.addEventListener('mouseenter', () => {
      if (infobulleEpinglee) return;
      cercle.classList.add('actif');
      infobulle.innerHTML = contenuApercu(p);
      infobulle.style.opacity = 1;
    });
    cercle.addEventListener('mousemove', (ev) => {
      if (infobulleEpinglee) return;
      positionnerInfobulle(ev);
    });
    cercle.addEventListener('mouseleave', () => {
      if (infobulleEpinglee) return;
      cercle.classList.remove('actif');
      infobulle.style.opacity = 0;
    });
    cercle.addEventListener('click', (ev) => {
      ev.stopPropagation();
      positionnerInfobulle(ev);
      infobulle.innerHTML = contenuDetaille(p) + '<span class="fermer-infobulle" title="Fermer">×</span>';
      infobulle.style.opacity = 1;
      infobulle.classList.add('epinglee');
      infobulleEpinglee = true;
      document.querySelectorAll('.point-frise.actif').forEach(c => c.classList.remove('actif'));
      cercle.classList.add('actif');
      infobulle.querySelector('.fermer-infobulle').addEventListener('click', fermerInfobulle);
    });
  });

  document.addEventListener('click', (ev) => {
    if (infobulleEpinglee && !infobulle.contains(ev.target) && !ev.target.classList.contains('point-frise')) {
      fermerInfobulle();
    }
  });

  const boutonTableau = document.getElementById('boutonTableau');
  boutonTableau.addEventListener('click', () => {
    const tableau = document.getElementById('tableauDetaille');
    const visible = tableau.classList.toggle('visible');
    boutonTableau.textContent = visible ? 'Masquer le tableau détaillé' : 'Afficher le tableau détaillé';
  });
</script>
</body></html>"""

html_final = (TEMPLATE_HTML
    .replace("__LARGEUR__", str(LARGEUR))
    .replace("__HAUTEUR__", str(HAUTEUR))
    .replace("__FRISE_SVG__", FRISE_SVG_CONTENU)
    .replace("__LIGNES_TABLEAU__", LIGNES_TABLEAU)
    .replace("__POINTS__", json.dumps(points, ensure_ascii=False))
    .replace("__LIBELLES__", json.dumps(LIBELLES_TECHNIQUE, ensure_ascii=False)))

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print("Frise écrite dans", CHEMIN_SORTIE)